# Notebook 07 — Personas & segments
Objectif : répondre à **"qui souffre de quoi ?"**

On s'appuie sur :
- les commentaires nettoyés (`outputs/comments_clean.parquet`)
- le score business par motif (Notebook 06)
- et, si présent : les hotspots contextuels (`outputs/block3_hotspots.csv`).


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import numpy as np
import pandas as pd

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 140)

print("OK ✅")

## 1) Charger la base + le score business (Notebook 06)

In [ ]:
def load_base_df() -> tuple[pd.DataFrame, str]:
    p = OUT_DIR / "comments_clean.parquet"
    if p.exists():
        return pd.read_parquet(p), str(p)
    p = OUT_DIR / "comments_clean.csv"
    if p.exists():
        return pd.read_csv(p), str(p)
    raise FileNotFoundError("Base introuvable. Lance Notebook 01 (ou mets outputs/comments_clean.parquet).")

df, used = load_base_df()
if "motif" in df.columns:
    df["motif"] = df["motif"].astype(str).str.strip().str.lower()
if "sentiment" in df.columns:
    df["sentiment"] = df["sentiment"].astype(str).str.strip().str.lower()
if "note" in df.columns:
    df["note"] = pd.to_numeric(df["note"], errors="coerce")

score_path = OUT_DIR / "block6_business_impact_motif.csv"
if not score_path.exists():
    raise FileNotFoundError("Score business introuvable. Lance d'abord le Notebook 06.")
score = pd.read_csv(score_path)
score["motif"] = score["motif"].astype(str).str.lower()

print("Base:", used, "| lignes:", len(df))
print("Score business:", score_path, "| motifs:", len(score))
score.head(5)

## 2) Pain index par segment (directement depuis la base)
On calcule un indicateur simple par segment :
`pain = volume * neg_share * note_penalty` (pondéré par le score business du motif)

In [ ]:
# jointure du score motif sur la base
df2 = df.merge(score[["motif","business_score_0_100","neg_share","note_mean"]], on="motif", how="left")

# note_penalty par ligne (si note absente -> 0)
if "note" in df2.columns:
    note_pen = (5 - df2["note"]).clip(lower=0)
else:
    note_pen = 0.0

# neg flag par ligne (si sentiment absente -> 0.5 neutre)
if "sentiment" in df2.columns:
    is_neg = (df2["sentiment"] == "neg").astype(float)
else:
    is_neg = 0.5

df2["pain_line"] = (1 + note_pen.fillna(0)) * (0.5 + is_neg)  # proxy
df2["pain_line"] = df2["pain_line"] * (df2["business_score_0_100"].fillna(50) / 100)

dims = [c for c in ["anciennete","formule","device","canal","region","support","contexte"] if c in df2.columns]
dims

### Top segments par dimension

In [ ]:
def segment_table(dim: str, topk: int = 15) -> pd.DataFrame:
    t = (
        df2.groupby(dim)
           .agg(
               n=("motif","size"),
               pain=("pain_line","sum"),
               avg_note=("note","mean"),
               neg_share=("sentiment", lambda s: (s=="neg").mean() if "sentiment" in df2.columns else np.nan),
           )
           .reset_index()
    )
    t["pain_per_comment"] = t["pain"] / t["n"].replace(0, np.nan)
    return t.sort_values("pain", ascending=False).head(topk)

segment_tables = {dim: segment_table(dim) for dim in dims}

# affiche un exemple
if dims:
    segment_tables[dims[0]].head(10)
else:
    print("Aucune dimension segment trouvée dans la base (anciennete/formule/device/canal/...).")

## 3) Hotspots (si dispo) : "où" ça se concentre
Si tu as `outputs/block3_hotspots.csv`, on ajoute une vue :
- motif + situation + dimension + value + n

In [ ]:
hotspots_path = OUT_DIR / "block3_hotspots.csv"
if hotspots_path.exists():
    hotspots = pd.read_csv(hotspots_path)
    for c in ["motif","situation","dimension","value"]:
        hotspots[c] = hotspots[c].astype(str).str.lower()
    hotspots = hotspots.merge(score[["motif","business_score_0_100"]], on="motif", how="left")
    hotspots["pain_proxy"] = hotspots["n"] * (hotspots["business_score_0_100"].fillna(50) / 100)

    top_hotspots = hotspots.sort_values("pain_proxy", ascending=False).head(25)
    print("Hotspots chargés ✅", hotspots_path, "| lignes:", len(hotspots))
    top_hotspots
else:
    hotspots = None
    print("Hotspots non trouvés (normal si Notebook 03 pas lancé).")

## 4) Personas (heuristiques simples)
On fabrique des *personas* lisibles, basés sur les segments les plus douloureux.
Ce ne sont pas des vérités : c'est une **mise en récit** pour aider l'action.


In [ ]:
personas = []

# Persona 1 : nouveaux clients (si anciennete existe)
if "anciennete" in df2.columns:
    t = segment_table("anciennete", topk=5)
    if len(t):
        top = t.iloc[0]
        personas.append({
            "persona": "Nouveau client",
            "segment": f"anciennete={top['anciennete']}",
            "signal": "pain élevé chez les nouveaux / récents",
            "action_hint": "simplifier le parcours + messages d'aide contextualisés",
        })

# Persona 2 : mobile iOS/Android (si device existe)
if "device" in df2.columns:
    t = segment_table("device", topk=5)
    if len(t):
        top = t.iloc[0]
        personas.append({
            "persona": "Mobile le plus impacté",
            "segment": f"device={top['device']}",
            "signal": "frictions concentrées sur un device",
            "action_hint": "tests QA ciblés + perf + crash recovery",
        })

# Persona 3 : canal dominant
if "canal" in df2.columns:
    t = segment_table("canal", topk=5)
    if len(t):
        top = t.iloc[0]
        personas.append({
            "persona": "Canal dominant",
            "segment": f"canal={top['canal']}",
            "signal": "un canal porte une grande partie de la douleur",
            "action_hint": "réduire le nombre d'étapes + clarifier les attentes",
        })

personas_df = pd.DataFrame(personas)
personas_df

## 5) Exports
- `block7_segments_<dimension>.csv`
- `block7_personas.csv`
- `block7_summary.json`


In [ ]:
exports = {}
for dim, tab in segment_tables.items():
    p = OUT_DIR / f"block7_segments_{dim}.csv"
    tab.to_csv(p, index=False)
    exports[dim] = str(p)

personas_path = OUT_DIR / "block7_personas.csv"
personas_df.to_csv(personas_path, index=False)
exports["personas"] = str(personas_path)

summary = {
    "base_source": used,
    "dims_used": dims,
    "exports": exports,
    "hotspots_used": bool(hotspots is not None),
}
with open(OUT_DIR / "block7_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Exports ✅")
for k,v in exports.items():
    print("-", k, "→", v)
print("-", OUT_DIR / "block7_summary.json")